# VeriGraph Live Demo
## GAN-Based Fact Verification System

**Presenter:** Marco  
**Duration:** 1 minute  
**Date:** March 2, 2026

This notebook demonstrates live fact-checking using our BERT-GAN model.

In [ ]:
# Setup: Load Model and Dependencies
import sys
sys.path.append('./macmini/venv/lib/python3.12/site-packages')

from factcheck import FactGAN
import spacy
import time

# Load spaCy for triplet extraction
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

# Load GAN model
print("Loading GAN discriminator...")
gan = FactGAN()
gan.load("./macmini/models/gan-model-staging-v3")

print("✅ Model loaded successfully!")
print(f"   Device: {gan.device}")
print(f"   Architecture: BERT-GAN-Swap v3")

## Helper Functions

In [ ]:
def extract_triplet(claim: str) -> tuple:
    """Extract (subject, predicate, object) from claim"""
    doc = nlp(claim)
    
    for token in doc:
        # Copular pattern: X is Y
        if token.pos_ == "AUX" and token.dep_ == "ROOT":
            subjects = [t for t in token.lefts if t.dep_ == "nsubj"]
            objects = [t for t in token.rights if t.dep_ in ["attr", "acomp"]]
            if subjects and objects:
                s = " ".join([t.text for t in subjects[0].subtree])
                p = token.text
                o = " ".join([t.text for t in objects[0].subtree])
                return (s, p, o)
        
        # Verbal pattern: X verb Y
        if token.pos_ == "VERB" and token.dep_ == "ROOT":
            subjects = [t for t in token.lefts if t.dep_ == "nsubj"]
            objects = [t for t in token.rights if t.dep_ == "dobj"]
            if subjects and objects:
                s = " ".join([t.text for t in subjects[0].subtree])
                p = token.lemma_
                o = " ".join([t.text for t in objects[0].subtree])
                return (s, p, o)
    
    return None


def verify_claim(claim: str):
    """Verify a factual claim using GAN discriminator"""
    print(f"\n{'='*70}")
    print(f"Claim: {claim}")
    print(f"{'='*70}")
    
    # Start timer
    start = time.time()
    
    # Extract triplet
    triplet = extract_triplet(claim)
    if not triplet:
        print("❌ Could not extract triplet from claim")
        return
    
    s, p, o = triplet
    print(f"Triplet: ({s}, {p}, {o})")
    
    # Run discriminator
    score = gan.discriminate_triplets([triplet])[0].item()
    
    # Determine verdict
    if score >= 0.7:
        verdict = "✅ SUPPORTED"
        confidence = score
    elif score <= 0.3:
        verdict = "❌ REFUTED"
        confidence = 1 - score
    else:
        verdict = "⚠️  NOT ENOUGH INFO"
        confidence = 0.5
    
    # End timer
    latency = (time.time() - start) * 1000
    
    # Print results
    print(f"\nGAN Score: {score:.4f}")
    print(f"Verdict: {verdict}")
    print(f"Confidence: {confidence:.2%}")
    print(f"Latency: {latency:.0f}ms")
    print(f"{'='*70}\n")

## 🎬 Live Demo - Example 1: Clear Fact

In [ ]:
verify_claim("Paris is the capital of France")

## 🎬 Live Demo - Example 2: Clear Misinformation

In [ ]:
verify_claim("London is the capital of Germany")

## 🎬 Live Demo - Example 3: Famous Person

In [ ]:
verify_claim("Barack Obama was born in Hawaii")

## 🎬 Interactive Demo - Test Your Own Claim!

Type any factual claim below and run the cell to verify it.

In [ ]:
# Test your own claim here!
your_claim = "Albert Einstein developed the theory of relativity"

verify_claim(your_claim)

## 📊 Model Performance Summary

| Metric | Value |
|--------|-------|
| **Test Accuracy** | 88.7% |
| **F1 Score** | 88.3% |
| **Average Latency** | 1.2 seconds |
| **Model Size** | 837 MB |
| **Parameters** | 110M (14M trainable) |

### Architecture Highlights:
- **Base Model:** BERT-base-uncased (Google)
- **Generator:** SwapGenerator (entity swapping)
- **Discriminator:** BERT + 2-layer classifier
- **Training Data:** 1.5M DBpedia triplets
- **Training Time:** 8 hours on RTX 3090

## 🔄 Batch Testing

Test multiple claims at once for efficiency comparison.

In [ ]:
# Batch test multiple claims
test_claims = [
    "The Eiffel Tower is in Paris",
    "Tokyo is the capital of Japan",
    "Rome is the capital of Spain",  # Fake!
    "Einstein played the violin",
    "The Earth is flat"  # Fake!
]

print("Running batch verification...")
print(f"\n{'='*70}")
print("BATCH RESULTS")
print(f"{'='*70}\n")

batch_start = time.time()
results = []

for claim in test_claims:
    triplet = extract_triplet(claim)
    if triplet:
        score = gan.discriminate_triplets([triplet])[0].item()
        
        if score >= 0.7:
            verdict = "✅ SUPPORTED"
            conf = score
        elif score <= 0.3:
            verdict = "❌ REFUTED"
            conf = 1 - score
        else:
            verdict = "⚠️  NOT ENOUGH INFO"
            conf = 0.5
        
        results.append({
            'claim': claim,
            'verdict': verdict,
            'confidence': conf,
            'score': score
        })

batch_time = (time.time() - batch_start) * 1000

# Print results table
for i, result in enumerate(results, 1):
    print(f"{i}. {result['claim']}")
    print(f"   {result['verdict']} ({result['confidence']:.0%}) [score: {result['score']:.3f}]")
    print()

print(f"{'='*70}")
print(f"Total batch time: {batch_time:.0f}ms")
print(f"Average per claim: {batch_time/len(test_claims):.0f}ms")
print(f"{'='*70}")

## 🎯 Confidence Distribution Analysis

Visualize how confident the model is across different claims.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Analyze confidence distribution from batch results
scores = [r['score'] for r in results]
confidences = [r['confidence'] for r in results]
verdicts = [r['verdict'] for r in results]

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: GAN scores
colors = ['green' if s >= 0.7 else 'red' if s <= 0.3 else 'orange' for s in scores]
ax1.bar(range(len(scores)), scores, color=colors, alpha=0.7)
ax1.axhline(y=0.7, color='g', linestyle='--', label='SUPPORTED threshold')
ax1.axhline(y=0.3, color='r', linestyle='--', label='REFUTED threshold')
ax1.set_xlabel('Claim Index')
ax1.set_ylabel('GAN Score')
ax1.set_title('GAN Discriminator Scores')
ax1.set_ylim(0, 1)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Confidence distribution
ax2.bar(range(len(confidences)), confidences, color=colors, alpha=0.7)
ax2.set_xlabel('Claim Index')
ax2.set_ylabel('Confidence')
ax2.set_title('Verdict Confidence')
ax2.set_ylim(0, 1)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📈 Statistics:")
print(f"   Average GAN score: {np.mean(scores):.3f}")
print(f"   Average confidence: {np.mean(confidences):.3f}")
print(f"   High confidence (>80%): {sum(1 for c in confidences if c > 0.8)}/{len(confidences)}")

## 🚀 Production Deployment Info

### Current Setup:
- **Server:** MacMini M1 (8GB RAM)
- **Database:** Neon PostgreSQL
- **Polling:** Every 1 second
- **Model Updates:** Check MLflow every 5 iterations
- **Uptime:** 99.8% (last 7 days)

### MLOps Pipeline:
```
Training → MLflow Registry → Quality Gates → Production
                                   ↓
                          (Min Accuracy: 75%)
                          (Min F1 Score: 70%)
```

### Version History:
| Version | Accuracy | F1 Score | Stage | Status |
|---------|----------|----------|-------|--------|
| v1 | 78.2% | 76.5% | Archived | Replaced |
| v2 | 85.3% | 84.7% | Production | Active |
| v3 | 88.7% | 88.3% | Staging | Testing |

## 💡 Key Takeaways

### What Makes This System Unique?

1. **Factual GAN Training**  
   Unlike traditional GANs (images), our discriminator learns to distinguish **factually correct** from **factually incorrect** statements.

2. **BERT Pre-Training Advantage**  
   Starting with BERT gives us 70%+ baseline accuracy. Fine-tuning adds 15%+.

3. **Fast Inference**  
   1.2s per claim (10x-25x faster than SOTA systems with comparable accuracy).

4. **Production-Ready MLOps**  
   Quality gates, automatic model updates, zero-downtime deployment.

5. **Explainable Results**  
   Returns triplets, entities, and confidence scores for human review.

### Future Enhancements:
- Multi-lingual support (French, Spanish, German)
- Video/audio claim verification
- Real-time browser extension
- Active learning for uncertain cases

## 🎓 Questions?

**Contact:**
- Paul: Development & MLOps
- Adam: Training & Architecture
- Marco: Prediction & Deployment

**Resources:**
- GitHub: https://github.com/MarcoSrhl/VeriGraph
- MLflow: https://dagshub.com/MarcoSrhl/NLP-Fact-checking
- Documentation: See PRESENTATION.md

---

**Thank you for watching the demo!** 🎉